# Grouping for Aggregation, Filtration, and Transformation

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 10, "display.max_rows", 10, "display.max_colwidth", 12)

## Introduction

### Defining an Aggregation

### How to do it...

In [2]:
flights = pd.read_csv("../data/flights.csv")
flights.head()

,MONTH,DAY,WEEKDAY,AIRLINE,ORG_AIR,...,DIST,SCHED_ARR,ARR_DELAY,DIVERTED,CANCELLED
0,1,1,4,WN,LAX,...,590,1905,65.0,0,0
1,1,1,4,UA,DEN,...,1452,1333,-13.0,0,0
2,1,1,4,MQ,DFW,...,641,1453,35.0,0,0
3,1,1,4,AA,DFW,...,1192,1935,-7.0,0,0
4,1,1,4,WN,LAX,...,1363,2225,39.0,0,0


In [3]:
(
    flights.groupby("AIRLINE")
    .agg({"ARR_DELAY": "mean"})
)

,ARR_DELAY
AIRLINE,
AA,5.542661
AS,-0.833333
B6,8.692593
DL,0.339691
EV,7.034580
...,...
OO,7.593463
UA,7.765755
US,1.681105


In [4]:
(
    flights.groupby("AIRLINE")["ARR_DELAY"]
    .agg("mean")
)

AIRLINE
AA    5.542661
AS   -0.833333
B6    8.692593
DL    0.339691
EV    7.034580
        ...   
OO    7.593463
UA    7.765755
US    1.681105
VX    5.348884
WN    6.397353
Name: ARR_DELAY, Length: 14, dtype: float64

In [5]:
(
    flights.groupby("AIRLINE")["ARR_DELAY"]
    .agg(np.mean)
)

/tmp/ipykernel_27992/3195466092.py:3: FutureWarning: The provided callable <function mean at 0xffffa86fe840> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(np.mean)


AIRLINE
AA    5.542661
AS   -0.833333
B6    8.692593
DL    0.339691
EV    7.034580
        ...   
OO    7.593463
UA    7.765755
US    1.681105
VX    5.348884
WN    6.397353
Name: ARR_DELAY, Length: 14, dtype: float64

In [6]:
(
    flights.groupby("AIRLINE")["ARR_DELAY"]
    .mean()
)

AIRLINE
AA    5.542661
AS   -0.833333
B6    8.692593
DL    0.339691
EV    7.034580
        ...   
OO    7.593463
UA    7.765755
US    1.681105
VX    5.348884
WN    6.397353
Name: ARR_DELAY, Length: 14, dtype: float64

### How it works...

In [7]:
grouped = flights.groupby("AIRLINE")
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

### There's more...

In [8]:
try:
    (flights.groupby("AIRLINE")["ARR_DELAY"].agg(np.sqrt))
except ValueError as e:
    print("ValueError: ", e)

ValueError:  Must produce aggregated value


/workspace/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Grouping and aggregating with multiple columns and functions

### How to do it...

In [9]:
(
    flights.groupby(["AIRLINE", "WEEKDAY"])["CANCELLED"]
    .agg("sum")
)

AIRLINE  WEEKDAY
AA       1          41
         2           9
         3          16
         4          20
         5          18
                    ..
WN       3          18
         4          10
         5           7
         6          10
         7           7
Name: CANCELLED, Length: 98, dtype: int64

In [10]:
(
    flights.groupby(["AIRLINE", "WEEKDAY"])[["CANCELLED", "DIVERTED"]]
    .agg(["sum", "mean"])
)

CANCELLED           DIVERTED          
                      sum      mean      sum      mean
AIRLINE WEEKDAY                                       
AA      1              41  0.032106        6  0.004699
        2               9  0.007341        2  0.001631
        3              16  0.011949        2  0.001494
        4              20  0.015004        5  0.003751
        5              18  0.014151        1  0.000786
...                   ...       ...      ...       ...
WN      3              18  0.014118        2  0.001569
        4              10  0.007911        4  0.003165
        5               7  0.005828        0  0.000000
        6              10  0.010132        3  0.003040
        7               7  0.006066        3  0.002600

[98 rows x 4 columns]

In [11]:
(
    flights.groupby(["ORG_AIR", "DEST_AIR"]).agg(
        {"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]}
    )
)

CANCELLED                   AIR_TIME            
                       sum      mean size        mean         var
ORG_AIR DEST_AIR                                                 
ATL     ABE              0  0.000000   31   96.387097   45.778495
        ABQ              0  0.000000   16  170.500000   87.866667
        ABY              0  0.000000   19   28.578947    6.590643
        ACY              0  0.000000    6   91.333333   11.466667
        AEX              0  0.000000   40   78.725000   47.332692
...                    ...       ...  ...         ...         ...
SFO     SNA              4  0.032787  122   64.059322   11.338331
        STL              0  0.000000   20  198.900000  101.042105
        SUN              0  0.000000   10   78.000000   25.777778
        TUS              0  0.000000   20  100.200000   35.221053
        XNA              0  0.000000    2  173.500000    0.500000

[1130 rows x 5 columns]

In [12]:
(
    flights.groupby(["ORG_AIR", "DEST_AIR"]).agg(
        sum_cancelled=pd.NamedAgg(column="CANCELLED", aggfunc="sum"),
        mean_cancelled=pd.NamedAgg(column="CANCELLED", aggfunc="mean"),
        size_cancelled=pd.NamedAgg(column="CANCELLED", aggfunc="size"),
        mean_air_time=pd.NamedAgg(column="AIR_TIME", aggfunc="mean"),
        var_air_time=pd.NamedAgg(column="AIR_TIME", aggfunc="var"),
    )
)

sum_cancelled  mean_cancelled  size_cancelled  \
ORG_AIR DEST_AIR                                                  
ATL     ABE                 0       0.000000              31      
        ABQ                 0       0.000000              16      
        ABY                 0       0.000000              19      
        ACY                 0       0.000000               6      
        AEX                 0       0.000000              40      
...                       ...            ...             ...      
SFO     SNA                 4       0.032787             122      
        STL                 0       0.000000              20      
        SUN                 0       0.000000              10      
        TUS                 0       0.000000              20      
        XNA                 0       0.000000               2      

                  mean_air_time  var_air_time  
ORG_AIR DEST_AIR                               
ATL     ABE         96.387097      45.778495   
        ABQ        170.500000      87.866667   
        ABY         28.578947       6.590643   
        ACY         91.333333      11.466667   
        AEX         78.725000      47.332692   
...                       ...            ...   
SFO     SNA         64.059322      11.338331   
        STL        198.900000     101.042105   
        SUN         78.000000      25.777778   
        TUS        100.200000      35.221053   
        XNA        173.500000       0.500000   

[1130 rows x 5 columns]

In [13]:
(
    flights.groupby(["ORG_AIR", "DEST_AIR"]).agg(
        {"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]}
    )
).columns

MultiIndex([('CANCELLED',  'sum'),
            ('CANCELLED', 'mean'),
            ('CANCELLED', 'size'),
            ( 'AIR_TIME', 'mean'),
            ( 'AIR_TIME',  'var')],
           )

In [14]:
(
    flights.groupby(["ORG_AIR", "DEST_AIR"]).agg(
        {"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]}
    )
).columns.to_flat_index()

Index([ ('CANCELLED', 'sum'), ('CANCELLED', 'mean'), ('CANCELLED', 'size'),
        ('AIR_TIME', 'mean'),   ('AIR_TIME', 'var')],
      dtype='object')

### How it works...

### There's more...

In [15]:
res = flights.groupby(["ORG_AIR", "DEST_AIR"]).agg(
    {"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]}
)
print(f"{type(res.columns)=}")

for x in res.columns:
    print(x)

print("__")
print(f"{type(res.columns.to_flat_index())=}")
for x in res.columns.to_flat_index():
    print(x)

type(res.columns)=<class 'pandas.core.indexes.multi.MultiIndex'>
('CANCELLED', 'sum')
('CANCELLED', 'mean')
('CANCELLED', 'size')
('AIR_TIME', 'mean')
('AIR_TIME', 'var')
__
type(res.columns.to_flat_index())=<class 'pandas.core.indexes.base.Index'>
('CANCELLED', 'sum')
('CANCELLED', 'mean')
('CANCELLED', 'size')
('AIR_TIME', 'mean')
('AIR_TIME', 'var')


In [16]:
res = flights.groupby(["ORG_AIR", "DEST_AIR"]).agg(
    {"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]}
)
res.columns = ["_".join(x) for x in res.columns.to_flat_index()]
res

CANCELLED_sum  CANCELLED_mean  CANCELLED_size  \
ORG_AIR DEST_AIR                                                  
ATL     ABE                 0       0.000000              31      
        ABQ                 0       0.000000              16      
        ABY                 0       0.000000              19      
        ACY                 0       0.000000               6      
        AEX                 0       0.000000              40      
...                       ...            ...             ...      
SFO     SNA                 4       0.032787             122      
        STL                 0       0.000000              20      
        SUN                 0       0.000000              10      
        TUS                 0       0.000000              20      
        XNA                 0       0.000000               2      

                  AIR_TIME_mean  AIR_TIME_var  
ORG_AIR DEST_AIR                               
ATL     ABE         96.387097      45.778495   
        ABQ        170.500000      87.866667   
        ABY         28.578947       6.590643   
        ACY         91.333333      11.466667   
        AEX         78.725000      47.332692   
...                       ...            ...   
SFO     SNA         64.059322      11.338331   
        STL        198.900000     101.042105   
        SUN         78.000000      25.777778   
        TUS        100.200000      35.221053   
        XNA        173.500000       0.500000   

[1130 rows x 5 columns]

In [17]:
def flatten_cols(df):
    df.columns = ["_".join(x) for x in df.columns] # df.columns.to_flat_index
    return df

In [18]:
res = (
    flights.groupby(["ORG_AIR", "DEST_AIR"])
    .agg({"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]})
    .pipe(flatten_cols)
)
res

CANCELLED_sum  CANCELLED_mean  CANCELLED_size  \
ORG_AIR DEST_AIR                                                  
ATL     ABE                 0       0.000000              31      
        ABQ                 0       0.000000              16      
        ABY                 0       0.000000              19      
        ACY                 0       0.000000               6      
        AEX                 0       0.000000              40      
...                       ...            ...             ...      
SFO     SNA                 4       0.032787             122      
        STL                 0       0.000000              20      
        SUN                 0       0.000000              10      
        TUS                 0       0.000000              20      
        XNA                 0       0.000000               2      

                  AIR_TIME_mean  AIR_TIME_var  
ORG_AIR DEST_AIR                               
ATL     ABE         96.387097      45.778495   
        ABQ        170.500000      87.866667   
        ABY         28.578947       6.590643   
        ACY         91.333333      11.466667   
        AEX         78.725000      47.332692   
...                       ...            ...   
SFO     SNA         64.059322      11.338331   
        STL        198.900000     101.042105   
        SUN         78.000000      25.777778   
        TUS        100.200000      35.221053   
        XNA        173.500000       0.500000   

[1130 rows x 5 columns]

> ```
> res = (
>     flights.assign(ORG_AIR=flights.ORG_AIR.astype("category"))
>     .groupby(["ORG_AIR", "DEST_AIR"])
>     .agg({"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]})
> )
> res 
> ```

/tmp/ipykernel_58778/4005112543.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["ORG_AIR", "DEST_AIR"])

In [19]:
# 2710 rows : with categorical data type, a Cartesian Product will be used. Risk of combinatoric explosion without `observed=True``
res = (
    flights.assign(ORG_AIR=flights.ORG_AIR.astype("category"))
    .groupby(["ORG_AIR", "DEST_AIR"])
    .agg({"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]})
)
res

/tmp/ipykernel_27992/152479011.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["ORG_AIR", "DEST_AIR"])


CANCELLED              AIR_TIME           
                       sum mean size        mean        var
ORG_AIR DEST_AIR                                           
ATL     ABE              0  0.0   31   96.387097  45.778495
        ABI              0  NaN    0         NaN        NaN
        ABQ              0  0.0   16  170.500000  87.866667
        ABR              0  NaN    0         NaN        NaN
        ABY              0  0.0   19   28.578947   6.590643
...                    ...  ...  ...         ...        ...
SFO     TYS              0  NaN    0         NaN        NaN
        VLD              0  NaN    0         NaN        NaN
        VPS              0  NaN    0         NaN        NaN
        XNA              0  0.0    2  173.500000   0.500000
        YUM              0  NaN    0         NaN        NaN

[2710 rows x 5 columns]

In [20]:
# 1130 rows with `observed=True`
res = (
    flights.assign(ORG_AIR=flights.ORG_AIR.astype("category"))
    .groupby(["ORG_AIR", "DEST_AIR"], observed=True)
    .agg({"CANCELLED": ["sum", "mean", "size"], "AIR_TIME": ["mean", "var"]})
)
res

CANCELLED                   AIR_TIME            
                       sum      mean size        mean         var
ORG_AIR DEST_AIR                                                 
ATL     ABE              0  0.000000   31   96.387097   45.778495
        ABQ              0  0.000000   16  170.500000   87.866667
        ABY              0  0.000000   19   28.578947    6.590643
        ACY              0  0.000000    6   91.333333   11.466667
        AEX              0  0.000000   40   78.725000   47.332692
...                    ...       ...  ...         ...         ...
SFO     SNA              4  0.032787  122   64.059322   11.338331
        STL              0  0.000000   20  198.900000  101.042105
        SUN              0  0.000000   10   78.000000   25.777778
        TUS              0  0.000000   20  100.200000   35.221053
        XNA              0  0.000000    2  173.500000    0.500000

[1130 rows x 5 columns]

## Removing the MultiIndex after grouping

In [21]:
flights = pd.read_csv("../data/flights.csv")
airline_info = (
    flights.groupby(["AIRLINE", "WEEKDAY"])
    .agg({"DIST": ["sum", "mean"], "ARR_DELAY": ["min", "max"]})
    .astype(int)
)
airline_info

DIST       ARR_DELAY     
                     sum  mean       min  max
AIRLINE WEEKDAY                              
AA      1        1455386  1139       -60  551
        2        1358256  1107       -52  725
        3        1496665  1117       -45  473
        4        1452394  1089       -46  349
        5        1427749  1122       -41  732
...                  ...   ...       ...  ...
WN      3         997213   782       -38  262
        4        1024854   810       -52  284
        5         981036   816       -44  244
        6         823946   834       -41  290
        7         945679   819       -45  261

[98 rows x 4 columns]

In [22]:
airline_info.columns.get_level_values(0)

Index(['DIST', 'DIST', 'ARR_DELAY', 'ARR_DELAY'], dtype='object')

In [23]:
airline_info.columns.get_level_values(1)

Index(['sum', 'mean', 'min', 'max'], dtype='object')

In [24]:
airline_info.columns.to_flat_index()

Index([('DIST', 'sum'), ('DIST', 'mean'), ('ARR_DELAY', 'min'),
       ('ARR_DELAY', 'max')],
      dtype='object')

In [25]:
airline_info.columns = ["_".join(x) for x in airline_info.columns.to_flat_index()]

In [26]:
airline_info

DIST_sum  DIST_mean  ARR_DELAY_min  ARR_DELAY_max
AIRLINE WEEKDAY                                                   
AA      1         1455386       1139          -60            551  
        2         1358256       1107          -52            725  
        3         1496665       1117          -45            473  
        4         1452394       1089          -46            349  
        5         1427749       1122          -41            732  
...                   ...        ...          ...            ...  
WN      3          997213        782          -38            262  
        4         1024854        810          -52            284  
        5          981036        816          -44            244  
        6          823946        834          -41            290  
        7          945679        819          -45            261  

[98 rows x 4 columns]

In [27]:
airline_info.reset_index()

,AIRLINE,WEEKDAY,DIST_sum,DIST_mean,ARR_DELAY_min,ARR_DELAY_max
0,AA,1,1455386,1139,-60,551
1,AA,2,1358256,1107,-52,725
2,AA,3,1496665,1117,-45,473
3,AA,4,1452394,1089,-46,349
4,AA,5,1427749,1122,-41,732
...,...,...,...,...,...,...
93,WN,3,997213,782,-38,262
94,WN,4,1024854,810,-52,284
95,WN,5,981036,816,-44,244
96,WN,6,823946,834,-41,290


In [28]:
(
    flights.groupby(["AIRLINE", "WEEKDAY"])
    .agg(
        dist_sum=pd.NamedAgg(column="DIST", aggfunc="sum"),
        dist_mean=pd.NamedAgg(column="DIST", aggfunc="mean"),
        arr_delay_min=pd.NamedAgg(column="ARR_DELAY", aggfunc="min"),
        arr_delay_max=pd.NamedAgg(column="ARR_DELAY", aggfunc="max"),
    )
    .astype(int)
    .reset_index()
)

,AIRLINE,WEEKDAY,dist_sum,dist_mean,arr_delay_min,arr_delay_max
0,AA,1,1455386,1139,-60,551
1,AA,2,1358256,1107,-52,725
2,AA,3,1496665,1117,-45,473
3,AA,4,1452394,1089,-46,349
4,AA,5,1427749,1122,-41,732
...,...,...,...,...,...,...
93,WN,3,997213,782,-38,262
94,WN,4,1024854,810,-52,284
95,WN,5,981036,816,-44,244
96,WN,6,823946,834,-41,290


### How it works...

### There's more...

In [29]:
(flights.groupby(["AIRLINE"], as_index=False)["DIST"].agg("mean").round(0))

,AIRLINE,DIST
0,AA,1114.0
1,AS,1066.0
2,B6,1772.0
3,DL,866.0
4,EV,460.0
...,...,...
9,OO,511.0
10,UA,1231.0
11,US,1181.0
12,VX,1240.0


## Grouping with a custom aggregation function

### How to do it...

In [30]:
college = pd.read_csv("../data/college.csv")
(
    college.groupby("STABBR")["UGDS"]
        .agg(["mean", "std"])
        .round(0)
)

,mean,std
STABBR,,
AK,2493.0,4052.0
AL,2790.0,4658.0
AR,1644.0,3143.0
AS,1276.0,NaN
AZ,4130.0,14894.0
...,...,...
VT,1513.0,2194.0
WA,2271.0,4124.0
WI,2655.0,4615.0


In [31]:
def max_deviation(s):
    std_score = (s - s.mean()) / s.std()
    return std_score.abs().max()

In [32]:
(
    college.groupby("STABBR")["UGDS"]
        .agg(max_deviation)
        .round(1)
)

STABBR
AK    2.6
AL    5.8
AR    6.3
AS    NaN
AZ    9.9
     ... 
VT    3.8
WA    6.6
WI    5.8
WV    7.2
WY    2.8
Name: UGDS, Length: 59, dtype: float64

### How it works...

### There's more...

In [33]:
(
    college.groupby("STABBR")[["UGDS", "SATVRMID", "SATMTMID"]]
    .agg(max_deviation)
    .round(1)
)

,UGDS,SATVRMID,SATMTMID
STABBR,,,
AK,2.6,NaN,NaN
AL,5.8,1.6,1.8
AR,6.3,2.2,2.3
AS,NaN,NaN,NaN
AZ,9.9,1.9,1.4
...,...,...,...
VT,3.8,1.9,1.9
WA,6.6,2.2,2.0
WI,5.8,2.4,2.2


In [34]:
(
    college.groupby(["STABBR", "RELAFFIL"])[["UGDS", "SATVRMID", "SATMTMID"]]
    .agg([max_deviation, "mean", "std"])
    .round(1)
)

UGDS                      SATVRMID               \
                max_deviation    mean     std max_deviation   mean   std   
STABBR RELAFFIL                                                            
AK     0                 2.1   3508.9  4539.5          NaN     NaN   NaN   
       1                 1.1    123.3   132.9          NaN   555.0   NaN   
AL     0                 5.2   3248.8  5102.4          1.6   514.9  56.5   
       1                 2.4    979.7   870.8          1.5   498.0  53.0   
AR     0                 5.8   1793.7  3401.6          1.9   481.1  37.9   
...                      ...      ...     ...          ...     ...   ...   
WI     0                 5.3   2879.1  5031.5          1.3   558.8  47.5   
       1                 3.4   1716.2  1934.6          2.1   500.1  66.0   
WV     0                 6.9   1873.9  6271.7          1.6   466.7  27.9   
       1                 1.3    716.4   503.6          1.9   485.7  14.6   
WY     0                 2.8   2244.4  2744.7          NaN   535.0   NaN   

                     SATMTMID               
                max_deviation   mean   std  
STABBR RELAFFIL                             
AK     0                 NaN     NaN   NaN  
       1                 NaN   503.0   NaN  
AL     0                 1.7   515.8  56.7  
       1                 1.4   485.6  61.4  
AR     0                 2.0   503.6  39.0  
...                      ...     ...   ...  
WI     0                 1.3   591.2  85.7  
       1                 1.8   526.6  42.5  
WV     0                 1.8   480.0  27.7  
       1                 1.7   484.8  17.7  
WY     0                 NaN   540.0   NaN  

[112 rows x 9 columns]

In [35]:
max_deviation.__name__

'max_deviation'

In [36]:
max_deviation.__name__ = "Max Deviation"
(
    college.groupby(["STABBR", "RELAFFIL"])[["UGDS", "SATVRMID", "SATMTMID"]]
    .agg([max_deviation, "mean", "std"])
    .round(1)
)

UGDS                      SATVRMID               \
                Max Deviation    mean     std Max Deviation   mean   std   
STABBR RELAFFIL                                                            
AK     0                 2.1   3508.9  4539.5          NaN     NaN   NaN   
       1                 1.1    123.3   132.9          NaN   555.0   NaN   
AL     0                 5.2   3248.8  5102.4          1.6   514.9  56.5   
       1                 2.4    979.7   870.8          1.5   498.0  53.0   
AR     0                 5.8   1793.7  3401.6          1.9   481.1  37.9   
...                      ...      ...     ...          ...     ...   ...   
WI     0                 5.3   2879.1  5031.5          1.3   558.8  47.5   
       1                 3.4   1716.2  1934.6          2.1   500.1  66.0   
WV     0                 6.9   1873.9  6271.7          1.6   466.7  27.9   
       1                 1.3    716.4   503.6          1.9   485.7  14.6   
WY     0                 2.8   2244.4  2744.7          NaN   535.0   NaN   

                     SATMTMID               
                Max Deviation   mean   std  
STABBR RELAFFIL                             
AK     0                 NaN     NaN   NaN  
       1                 NaN   503.0   NaN  
AL     0                 1.7   515.8  56.7  
       1                 1.4   485.6  61.4  
AR     0                 2.0   503.6  39.0  
...                      ...     ...   ...  
WI     0                 1.3   591.2  85.7  
       1                 1.8   526.6  42.5  
WV     0                 1.8   480.0  27.7  
       1                 1.7   484.8  17.7  
WY     0                 NaN   540.0   NaN  

[112 rows x 9 columns]

## Customizing aggregating functions with *args and **kwargs

### How to do it...

In [37]:
def pct_between_1_3k(s):
    return s.between(1_000, 3_000).mean() * 100

In [38]:
(
    college.groupby(["STABBR", "RELAFFIL"])["UGDS"]
        .agg(pct_between_1_3k)
        .round(1)
)

STABBR  RELAFFIL
AK      0           14.3
        1            0.0
AL      0           23.6
        1           33.3
AR      0           27.9
                    ... 
WI      0           13.8
        1           36.0
WV      0           24.6
        1           37.5
WY      0           54.5
Name: UGDS, Length: 112, dtype: float64

In [39]:
def pct_between(s, low, high):
    return s.between(low, high).mean() * 100

In [40]:
(
    college.groupby(["STABBR", "RELAFFIL"])["UGDS"]
    .agg(pct_between, 1_000, 10_000)
    .round(1)
)

STABBR  RELAFFIL
AK      0           42.9
        1            0.0
AL      0           45.8
        1           37.5
AR      0           39.7
                    ... 
WI      0           31.0
        1           44.0
WV      0           29.2
        1           37.5
WY      0           72.7
Name: UGDS, Length: 112, dtype: float64

### How it works...

### There's more...

In [41]:
def between_n_m(n, m):
    def wrapper(ser):
        return pct_between(ser, n, m)

    wrapper.__name__ = f"between_{n}_{m}"
    return wrapper

In [42]:
(
    college.groupby(["STABBR", "RELAFFIL"])["UGDS"]
    .agg([between_n_m(1_000, 10_000), "max", "mean"])
    .round(1)
)

between_1000_10000      max    mean
STABBR RELAFFIL                                     
AK     0                42.9         12865.0  3508.9
       1                 0.0           275.0   123.3
AL     0                45.8         29851.0  3248.8
       1                37.5          3033.0   979.7
AR     0                39.7         21405.0  1793.7
...                      ...             ...     ...
WI     0                31.0         29302.0  2879.1
       1                44.0          8212.0  1716.2
WV     0                29.2         44924.0  1873.9
       1                37.5          1375.0   716.4
WY     0                72.7          9910.0  2244.4

[112 rows x 3 columns]

## Examining the groupby object

### How to do it...

In [43]:
college = pd.read_csv("../data/college.csv")
grouped = college.groupby(["STABBR", "RELAFFIL"])
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

In [44]:
print([attr for attr in dir(grouped) if not attr.startswith("_")])

['CITY', 'CURROPER', 'DISTANCEONLY', 'GRAD_DEBT_MDN_SUPP', 'HBCU', 'INSTNM', 'MD_EARN_WNE_P10', 'MENONLY', 'PCTFLOAN', 'PCTPELL', 'PPTUG_EF', 'RELAFFIL', 'SATMTMID', 'SATVRMID', 'STABBR', 'UG25ABV', 'UGDS', 'UGDS_2MOR', 'UGDS_AIAN', 'UGDS_ASIAN', 'UGDS_BLACK', 'UGDS_HISP', 'UGDS_NHPI', 'UGDS_NRA', 'UGDS_UNKN', 'UGDS_WHITE', 'WOMENONLY', 'agg', 'aggregate', 'all', 'any', 'apply', 'bfill', 'boxplot', 'corr', 'corrwith', 'count', 'cov', 'cumcount', 'cummax', 'cummin', 'cumprod', 'cumsum', 'describe', 'diff', 'dtypes', 'ewm', 'expanding', 'ffill', 'fillna', 'filter', 'first', 'get_group', 'groups', 'head', 'hist', 'idxmax', 'idxmin', 'indices', 'last', 'max', 'mean', 'median', 'min', 'ndim', 'ngroup', 'ngroups', 'nth', 'nunique', 'ohlc', 'pct_change', 'pipe', 'plot', 'prod', 'quantile', 'rank', 'resample', 'rolling', 'sample', 'sem', 'shift', 'size', 'skew', 'std', 'sum', 'tail', 'take', 'transform', 'value_counts', 'var']


In [45]:
grouped.ngroups

112

In [46]:
groups = list(grouped.groups)
groups[:6]

[('AK', 0), ('AK', 1), ('AL', 0), ('AL', 1), ('AR', 0), ('AR', 1)]

In [47]:
grouped.get_group(("FL", 1))

,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
712,The Bapt...,Graceville,FL,0.0,0.0,...,0.5878,0.5602,0.3531,30800,20052
713,Barry Un...,Miami,FL,0.0,0.0,...,0.5045,0.6733,0.4361,44100,28250
714,Gooding ...,Panama City,FL,0.0,0.0,...,NaN,NaN,NaN,NaN,PrivacyS...
715,Bethune-...,Daytona ...,FL,1.0,0.0,...,0.7758,0.8867,0.0647,29400,36250
724,Johnson ...,Kissimmee,FL,0.0,0.0,...,0.6689,0.7384,0.2185,26300,20199
...,...,...,...,...,...,...,...,...,...,...,...
7486,Strayer ...,Coral Sp...,FL,NaN,NaN,...,NaN,NaN,NaN,49200,36173.5
7487,Strayer ...,Fort Lau...,FL,NaN,NaN,...,NaN,NaN,NaN,49200,36173.5
7488,Strayer ...,Miramar,FL,NaN,NaN,...,NaN,NaN,NaN,49200,36173.5
7489,Strayer ...,Miami,FL,NaN,NaN,...,NaN,NaN,NaN,49200,36173.5


In [48]:
for name, group in list(grouped)[:10]:
    print(name)
    display(group.head(3))

('AK', 0)


,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
60,Universi...,Anchorage,AK,0.0,0.0,...,0.2385,0.2647,0.4386,42500,19449.5
62,Universi...,Fairbanks,AK,0.0,0.0,...,0.2263,0.2550,0.4519,36200,19355
63,Universi...,Juneau,AK,0.0,0.0,...,0.1769,0.1996,0.5550,37400,16875


('AK', 1)


,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
61,Alaska B...,Palmer,AK,0.0,0.0,...,0.3571,0.2857,0.4286,NaN,PrivacyS...
64,Alaska P...,Anchorage,AK,0.0,0.0,...,0.3152,0.5297,0.4910,47000,23250
5417,Alaska C...,Soldotna,AK,0.0,0.0,...,0.8868,0.6792,0.2264,NaN,PrivacyS...


('AL', 0)


...

,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
69,Collins ...,Phoenix,AZ,0.0,0.0,...,0.7205,0.8228,0.4764,25700,47000
71,Empire B...,Tucson,AZ,0.0,0.0,...,0.7962,0.6615,0.4229,18200,9833
72,Thunderb...,Glendale,AZ,0.0,0.0,...,0.0000,0.0000,0.0000,118900,PrivacyS...


('AZ', 1)


,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
68,Everest ...,Phoenix,AZ,0.0,0.0,...,0.8291,0.7151,0.6700,28600,9500
70,Empire B...,Phoenix,AZ,0.0,0.0,...,0.6349,0.5873,0.4651,17800,9588
73,American...,Phoenix,AZ,0.0,0.0,...,0.7500,0.5375,0.4684,PrivacyS...,PrivacyS...


('CA', 0)


,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
192,Academy ...,San Fran...,CA,0.0,0.0,...,0.4008,0.5524,0.4043,36000,35093
193,ITT Tech...,Rancho C...,CA,0.0,0.0,...,0.7137,0.7667,0.7235,38800,25827.5
194,Academy ...,Oakland,CA,0.0,0.0,...,NaN,NaN,NaN,NaN,PrivacyS...


In [49]:
for name, group in grouped:
    print(name)
    print(group)
    break

('AK', 0)
           INSTNM       CITY STABBR  HBCU  MENONLY  ...  PCTPELL  PCTFLOAN  \
60    Universi...  Anchorage     AK   0.0      0.0  ...   0.2385    0.2647   
62    Universi...  Fairbanks     AK   0.0      0.0  ...   0.2263    0.2550   
63    Universi...     Juneau     AK   0.0      0.0  ...   0.1769    0.1996   
65    AVTEC-Al...     Seward     AK   0.0      0.0  ...   0.0737    0.0664   
66    Charter ...  Anchorage     AK   0.0      0.0  ...   0.8307    0.7503   
67    Alaska C...  Anchorage     AK   0.0      0.0  ...   0.7078    0.7860   
5171  Ilisagvi...     Barrow     AK   0.0      0.0  ...   0.1323    0.0000   

      UG25ABV  MD_EARN_WNE_P10  GRAD_DEBT_MDN_SUPP  
60     0.4386        42500          19449.5         
62     0.4519        36200            19355         
63     0.5550        37400            16875         
65     0.7127        33500      PrivacyS...         
66     0.5472        39200            13875         
67     0.5612        28700             8994    

In [50]:
grouped.head(2)

,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
0,Alabama ...,Normal,AL,1.0,0.0,...,0.7356,0.8284,0.1049,30300,33888
1,Universi...,Birmingham,AL,0.0,0.0,...,0.3460,0.5214,0.2422,39700,21941.5
2,Amridge ...,Montgomery,AL,0.0,0.0,...,0.6801,0.7795,0.8540,40100,23370
10,Birmingh...,Birmingham,AL,0.0,0.0,...,0.1920,0.4809,0.0152,44200,27000
43,Prince I...,Elmhurst,IL,0.0,0.0,...,0.7857,0.9375,0.6569,PrivacyS...,20992
...,...,...,...,...,...,...,...,...,...,...,...
5289,Pacific ...,Mangilao,GU,0.0,0.0,...,0.9730,0.0000,0.2533,PrivacyS...,PrivacyS...
6439,Touro Un...,Henderson,NV,0.0,0.0,...,0.0000,0.2000,0.4000,NaN,PrivacyS...
7352,Marinell...,Henderson,NV,NaN,NaN,...,NaN,NaN,NaN,21200,9796.5
7404,Universi...,St. Croix,VI,NaN,NaN,...,NaN,NaN,NaN,31800,15150


In [51]:
grouped.nth([1, -1])

,INSTNM,CITY,STABBR,HBCU,MENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
1,Universi...,Birmingham,AL,0.0,0.0,...,0.3460,0.5214,0.2422,39700,21941.5
10,Birmingh...,Birmingham,AL,0.0,0.0,...,0.1920,0.4809,0.0152,44200,27000
62,Universi...,Fairbanks,AK,0.0,0.0,...,0.2263,0.2550,0.4519,36200,19355
64,Alaska P...,Anchorage,AK,0.0,0.0,...,0.3152,0.5297,0.4910,47000,23250
70,Empire B...,Phoenix,AZ,0.0,0.0,...,0.6349,0.5873,0.4651,17800,9588
...,...,...,...,...,...,...,...,...,...,...,...
7519,Strayer ...,North Ch...,SC,NaN,NaN,...,NaN,NaN,NaN,49200,36173.5
7531,Rasmusse...,Overland...,KS,NaN,NaN,...,NaN,NaN,NaN,NaN,21163
7532,National...,Highland...,OH,NaN,NaN,...,NaN,NaN,NaN,NaN,6333
7533,Bay Area...,San Jose,CA,NaN,NaN,...,NaN,NaN,NaN,NaN,PrivacyS...


## Filtering for states with a minority majority

In [52]:
college = pd.read_csv("../data/college.csv", index_col="INSTNM")
grouped = college.groupby("STABBR")
print(grouped.ngroups)
print(college.STABBR.nunique())

59
59


In [53]:
def check_minority(
        df:pd.DataFrame,
        threshold: float) -> bool:
    minority_pct = 1 - df["UGDS_WHITE"]
    total_minority = (df["UGDS"] * minority_pct).sum()
    total_ugds = df["UGDS"].sum()
    total_minority_pct = total_minority / total_ugds
    return total_minority_pct > threshold

In [54]:
college_filtered = grouped.filter(check_minority, threshold=.5)
college_filtered

,CITY,STABBR,HBCU,MENONLY,WOMENONLY,...,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
INSTNM,,,,,,,,,,,
Everest College-Phoenix,Phoenix,AZ,0.0,0.0,0.0,...,0.8291,0.7151,0.6700,28600,9500
Collins College,Phoenix,AZ,0.0,0.0,0.0,...,0.7205,0.8228,0.4764,25700,47000
Empire Beauty School-Paradise Valley,Phoenix,AZ,0.0,0.0,0.0,...,0.6349,0.5873,0.4651,17800,9588
Empire Beauty School-Tucson,Tucson,AZ,0.0,0.0,0.0,...,0.7962,0.6615,0.4229,18200,9833
Thunderbird School of Global Management,Glendale,AZ,0.0,0.0,0.0,...,0.0000,0.0000,0.0000,118900,PrivacyS...
...,...,...,...,...,...,...,...,...,...,...,...
WestMed College - Merced,Merced,CA,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,15623.5
Vantage College,El Paso,TX,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,9500
SAE Institute of Technology San Francisco,Emeryville,CA,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,9500


In [55]:
print(f"{college.shape=}")
print(f"{college_filtered.shape=}")
print(f"{college_filtered.STABBR.nunique()=}")

college.shape=(7535, 26)
college_filtered.shape=(3028, 26)
college_filtered.STABBR.nunique()=20


In [56]:
college_filtered_20 = grouped.filter(check_minority, threshold=.2)
print(f"{college_filtered_20.shape=}")
print(f"{college_filtered_20.STABBR.nunique()=}")

college_filtered_70 = grouped.filter(check_minority, threshold=.7)
print(f"{college_filtered_70.shape=}")
print(f"{college_filtered_70.STABBR.nunique()=}")

college_filtered_20.shape=(7461, 26)
college_filtered_20.STABBR.nunique()=57
college_filtered_70.shape=(957, 26)
college_filtered_70.STABBR.nunique()=10


## Transforming though a weight loss bet

In [57]:
weight_loss = pd.read_csv("../data/weight_loss.csv")
weight_loss.query("Month == 'Jan'")

,Name,Month,Week,Weight
0,Bob,Jan,Week 1,291
1,Amy,Jan,Week 1,197
2,Bob,Jan,Week 2,288
3,Amy,Jan,Week 2,189
4,Bob,Jan,Week 3,283
5,Amy,Jan,Week 3,189
6,Bob,Jan,Week 4,283
7,Amy,Jan,Week 4,190


In [58]:
# compare values from 1st week and last week of each month to check who won the bet on a given month
def percent_loss(s: pd.Series) -> pd.Series:
    return ((s - s.iloc[0]) / s.iloc[0]) * 100

In [59]:
(
    weight_loss.query("Name == 'Bob' and Month=='Jan'")
        ["Weight"]
        .pipe(percent_loss)
)

0    0.000000
2   -1.030928
4   -2.749141
6   -2.749141
Name: Weight, dtype: float64

In [60]:
(
    weight_loss.groupby(["Name", "Month"])
        ["Weight"]
        .transform(percent_loss)
)

0     0.000000
1     0.000000
2    -1.030928
3    -4.060914
4    -2.749141
        ...   
27   -3.529412
28   -3.065134
29   -3.529412
30   -4.214559
31   -5.294118
Name: Weight, Length: 32, dtype: float64

In [61]:
weight_loss.groupby(["Name", "Month"]).nunique()

Week  Weight
Name Month              
Amy  Apr       4       3
     Feb       4       4
     Jan       4       3
     Mar       4       2
Bob  Apr       4       4
     Feb       4       3
     Jan       4       3
     Mar       4       4

In [62]:
weight_loss.groupby(["Name", "Month"]).nunique().sum()

Week      32
Weight    26
dtype: int64

In [63]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                .round(1)
                )
            )
        .query("Name=='Bob' and Month in ['Jan', 'Feb']")
)

,Name,Month,Week,Weight,percent_loss
0,Bob,Jan,Week 1,291,0.0
2,Bob,Jan,Week 2,288,-1.0
4,Bob,Jan,Week 3,283,-2.7
6,Bob,Jan,Week 4,283,-2.7
8,Bob,Feb,Week 1,283,0.0
10,Bob,Feb,Week 2,275,-2.8
12,Bob,Feb,Week 3,268,-5.3
14,Bob,Feb,Week 4,268,-5.3


In [64]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                .round(1)
                )
            )
        .query("Week=='Week 4'")
)

,Name,Month,Week,Weight,percent_loss
6,Bob,Jan,Week 4,283,-2.7
7,Amy,Jan,Week 4,190,-3.6
14,Bob,Feb,Week 4,268,-5.3
15,Amy,Feb,Week 4,173,-8.9
22,Bob,Mar,Week 4,261,-2.6
23,Amy,Mar,Week 4,170,-1.7
30,Bob,Apr,Week 4,250,-4.2
31,Amy,Apr,Week 4,161,-5.3


In [65]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                .round(1)
                )
            )
        .query("Week=='Week 4'")
        .pivot(index="Month", columns="Name", values="percent_loss")
)

Name,Amy,Bob
Month,,
Apr,-5.3,-4.2
Feb,-8.9,-5.3
Jan,-3.6,-2.7
Mar,-1.7,-2.6


In [66]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                .round(1)
                )
            )
        .query("Week=='Week 4'")
        .pivot(index="Month", columns="Name", values="percent_loss")
        .assign(winner=lambda df_: np.where(df_.Amy < df_.Bob, "Amy", "Bob"))
)

Name,Amy,Bob,winner
Month,,,
Apr,-5.3,-4.2,Amy
Feb,-8.9,-5.3,Amy
Jan,-3.6,-2.7,Amy
Mar,-1.7,-2.6,Bob


In [67]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                )
            )
        .query("Week=='Week 4'")
        .pivot(index="Month", columns="Name", values="percent_loss")
        .assign(winner=lambda df_: np.where(df_.Amy < df_.Bob, "Amy", "Bob"))
        .style
        .format('{0:,.1f}', subset=["Amy", "Bob"])
        .highlight_min(subset=["Amy", "Bob"], axis=1)
)

Name,Amy,Bob,winner
Month,,,
Apr,-5.3,-4.2,Amy
Feb,-8.9,-5.3,Amy
Jan,-3.6,-2.7,Amy
Mar,-1.7,-2.6,Bob


In [68]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                )
            )
        .query("Week=='Week 4'")
        .pivot(index="Month", columns="Name", values="percent_loss")
        .assign(winner=lambda df_: np.where(df_.Amy < df_.Bob, "Amy", "Bob"))
        .winner.value_counts()
)

winner
Amy    3
Bob    1
Name: count, dtype: int64

In [69]:
(
    weight_loss.assign(percent_loss=(
            weight_loss.groupby(["Name", "Month"])
                ["Weight"]
                .transform(percent_loss)
                .round(1)
                )
            )
        .query("Week=='Week 4'")
        .groupby(["Month", "Name"])
        ["percent_loss"]
        .first()
        .unstack()
)

Name,Amy,Bob
Month,,
Apr,-5.3,-4.2
Feb,-8.9,-5.3
Jan,-3.6,-2.7
Mar,-1.7,-2.6


## Calculating weighted mean SAT scores per state with apply

In [70]:
college = pd.read_csv("../data/college.csv", index_col="INSTNM")
subset = ["UGDS", "SATMTMID", "SATVRMID"]
college2 = college.dropna(subset=subset) # we'll create a weighted average, so can't afford to divide by zero or NaN

print(f"{college.shape=}")
print(f"{college2.shape=}")

college.shape=(7535, 26)
college2.shape=(1184, 26)


In [71]:
def weighted_math_average(df):
    weighted_math = df['UGDS'] * df['SATMTMID']
    return int(weighted_math.sum() / df['UGDS'].sum())

In [72]:
college2.groupby('STABBR').apply(weighted_math_average)

/tmp/ipykernel_27992/4114906302.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  college2.groupby('STABBR').apply(weighted_math_average)


STABBR
AK    503
AL    536
AR    529
AZ    569
CA    564
     ... 
VT    566
WA    555
WI    593
WV    500
WY    540
Length: 53, dtype: int64

In [73]:
(college2
    .groupby('STABBR')
    .agg(weighted_math_average) # agg has no access to 'UGDS' columns
)

KeyError: 'UGDS'

In [74]:
def weighted_average(df):
   weight_m = df['UGDS'] * df['SATMTMID']
   weight_v = df['UGDS'] * df['SATVRMID']
   wm_avg = weight_m.sum() / df['UGDS'].sum()
   wv_avg = weight_v.sum() / df['UGDS'].sum()
   data = {'w_math_avg': wm_avg,
           'w_verbal_avg': wv_avg,
           'math_avg': df['SATMTMID'].mean(),
           'verbal_avg': df['SATVRMID'].mean(),
           'count': len(df)
   }
   return pd.Series(data)


(college2
    .groupby('STABBR')
    .apply(weighted_average)
    .astype(int)
)

/tmp/ipykernel_27992/969352952.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_average)


,w_math_avg,w_verbal_avg,math_avg,verbal_avg,count
STABBR,,,,,
AK,503,555,503,555,1
AL,536,533,504,508,21
AR,529,504,515,491,16
AZ,569,557,536,538,6
CA,564,539,562,549,72
...,...,...,...,...,...
VT,566,564,526,527,8
WA,555,541,551,548,18
WI,593,556,545,516,14


In [75]:
(college
    .groupby('STABBR')
    .apply(weighted_average)
)

/tmp/ipykernel_27992/2068887078.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_average)


,w_math_avg,w_verbal_avg,math_avg,verbal_avg,count
STABBR,,,,,
AK,5.548091,6.121651,503.000000,555.000000,10.0
AL,261.895658,260.550109,504.285714,508.476190,96.0
AR,301.054792,287.264872,515.937500,491.875000,86.0
AS,0.000000,0.000000,NaN,NaN,1.0
AZ,61.815821,60.511712,536.666667,538.333333,133.0
...,...,...,...,...,...
VT,389.967094,388.696848,526.875000,527.500000,27.0
WA,274.885878,267.880280,551.222222,548.333333,123.0
WI,153.803086,144.160115,545.071429,516.857143,112.0


In [76]:
# let's add Geometric Mean and Harmonic Mean

from scipy.stats import gmean, hmean

def calculate_means(df):
    df_means = pd.DataFrame(index=['Arithmetic', 'Weighted',
                                   'Geometric', 'Harmonic'])
    cols = ['SATMTMID', 'SATVRMID']
    for col in cols:
        arithmetic = df[col].mean()
        weighted = np.average(df[col], weights=df['UGDS'])
        geometric = gmean(df[col])
        harmonic = hmean(df[col])
        df_means[col] = [arithmetic, weighted,
                         geometric, harmonic]
    df_means['count'] = len(df)
    return df_means.astype(int)

(college2
    .groupby('STABBR')
    .apply(calculate_means)
)

/tmp/ipykernel_27992/4014203651.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_means)


SATMTMID  SATVRMID  count
STABBR                                      
AK     Arithmetic       503       555      1
       Weighted         503       555      1
       Geometric        503       555      1
       Harmonic         503       555      1
AL     Arithmetic       504       508     21
...                     ...       ...    ...
WV     Harmonic         480       472     17
WY     Arithmetic       540       535      1
       Weighted         540       535      1
       Geometric        540       534      1
       Harmonic         540       535      1

[212 rows x 3 columns]

## Grouping by continuous variables

In [77]:
flights = pd.read_csv('../data/flights.csv')
flights

,MONTH,DAY,WEEKDAY,AIRLINE,ORG_AIR,...,DIST,SCHED_ARR,ARR_DELAY,DIVERTED,CANCELLED
0,1,1,4,WN,LAX,...,590,1905,65.0,0,0
1,1,1,4,UA,DEN,...,1452,1333,-13.0,0,0
2,1,1,4,MQ,DFW,...,641,1453,35.0,0,0
3,1,1,4,AA,DFW,...,1192,1935,-7.0,0,0
4,1,1,4,WN,LAX,...,1363,2225,39.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
58487,12,31,4,AA,SFO,...,1464,1045,-19.0,0,0
58488,12,31,4,F9,LAS,...,414,2050,4.0,0,0
58489,12,31,4,OO,SFO,...,262,1956,-5.0,0,0
58490,12,31,4,WN,MSP,...,907,855,34.0,0,0


In [78]:
bins = [-np.inf, 200, 500, 1000, 2000, np.inf]
cuts = pd.cut(flights['DIST'], bins=bins)
cuts

0        (500.0, ...
1        (1000.0,...
2        (500.0, ...
3        (1000.0,...
4        (1000.0,...
            ...     
58487    (1000.0,...
58488    (200.0, ...
58489    (200.0, ...
58490    (500.0, ...
58491    (500.0, ...
Name: DIST, Length: 58492, dtype: category
Categories (5, interval[float64, right]): [(-inf, 2... < (200.0, ... < (500.0, ... < (1000.0,... < (2000.0,...]

In [79]:
cuts.value_counts()

DIST
(500.0, 1000.0]     20659
(200.0, 500.0]      15874
(1000.0, 2000.0]    14186
(2000.0, inf]        4054
(-inf, 200.0]        3719
Name: count, dtype: int64

In [80]:
(flights
    .groupby(cuts)
    ['AIRLINE']
    .value_counts(normalize=True)
    .round(3)
)

/tmp/ipykernel_27992/1419780275.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(cuts)


DIST           AIRLINE
(-inf, 200.0]  OO         0.326
               EV         0.289
               MQ         0.211
               DL         0.086
               AA         0.052
                          ...  
(2000.0, inf]  AS         0.012
               F9         0.004
               EV         0.000
               MQ         0.000
               OO         0.000
Name: proportion, Length: 70, dtype: float64

In [81]:
(flights
  .groupby(cuts)
  ['AIR_TIME']
  .quantile(q=[.25, .5, .75])
  .div(60) # airtime is in minutes so divide by 60 to get hours
  .round(2)
)

/tmp/ipykernel_27992/2372627610.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(cuts)


DIST                  
(-inf, 200.0]     0.25    0.43
                  0.50    0.50
                  0.75    0.57
(200.0, 500.0]    0.25    0.77
                  0.50    0.92
                          ... 
(1000.0, 2000.0]  0.50    2.93
                  0.75    3.40
(2000.0, inf]     0.25    4.30
                  0.50    4.70
                  0.75    5.03
Name: AIR_TIME, Length: 15, dtype: float64

In [82]:
labels=['Under an Hour', '1 Hour', '1-2 Hours',
        '2-4 Hours', '4+ Hours']

cuts2 = pd.cut(flights['DIST'], bins=bins, labels=labels) # seems WRONG, using distance bins and time labels

(flights
   .groupby(cuts2)
   ['AIRLINE']
   .value_counts(normalize=True)
   .round(3)
   #.unstack()
)

/tmp/ipykernel_27992/3465402191.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(cuts2)


DIST           AIRLINE
Under an Hour  OO         0.326
               EV         0.289
               MQ         0.211
               DL         0.086
               AA         0.052
                          ...  
4+ Hours       AS         0.012
               F9         0.004
               EV         0.000
               MQ         0.000
               OO         0.000
Name: proportion, Length: 70, dtype: float64

## Counting the total number of flights between cities

In [83]:
flights = pd.read_csv('../data/flights.csv')
flights_ct = flights.groupby(['ORG_AIR', 'DEST_AIR']).size()
flights_ct

ORG_AIR  DEST_AIR
ATL      ABE          31
         ABQ          16
         ABY          19
         ACY           6
         AEX          40
                    ... 
SFO      SNA         122
         STL          20
         SUN          10
         TUS          20
         XNA           2
Length: 1130, dtype: int64

In [84]:
flights_ct.loc[[('ATL', 'IAH'), ('IAH', 'ATL')]]

ORG_AIR  DEST_AIR
ATL      IAH         121
IAH      ATL         148
dtype: int64

In [85]:
flights[['ORG_AIR', 'DEST_AIR']]

,ORG_AIR,DEST_AIR
0,LAX,SLC
1,DEN,IAD
2,DFW,VPS
3,DFW,DCA
4,LAX,MCI
...,...,...
58487,SFO,DFW
58488,LAS,SFO
58489,SFO,SBA
58490,MSP,ATL


In [86]:
 #reset index to avoid that columns be realigned after the application of the function

f_part3 = (flights  # doctest: +SKIP
  [['ORG_AIR', 'DEST_AIR']]
  .apply(lambda ser:
         ser.sort_values().reset_index(drop=True),
         axis='columns')
)
f_part3

,0,1
0,LAX,SLC
1,DEN,IAD
2,DFW,VPS
3,DCA,DFW
4,LAX,MCI
...,...,...
58487,DFW,SFO
58488,LAS,SFO
58489,SBA,SFO
58490,ATL,MSP


In [87]:
rename_dict = {0:'AIR1', 1:'AIR2'}
(flights     # doctest: +SKIP
  [['ORG_AIR', 'DEST_AIR']]
  .apply(lambda ser:
         ser.sort_values().reset_index(drop=True),
         axis='columns')
  .rename(columns=rename_dict)
  .groupby(['AIR1', 'AIR2'])
  .size()
)

AIR1  AIR2
ABE   ATL      31
      ORD      24
ABI   DFW      74
ABQ   ATL      16
      DEN      46
             ... 
SFO   SNA     122
      STL      20
      SUN      10
      TUS      20
      XNA       2
Length: 1085, dtype: int64

In [89]:
%%timeit
(flights     # doctest: +SKIP
  [['ORG_AIR', 'DEST_AIR']]
  .apply(lambda ser:
         ser.sort_values().reset_index(drop=True),
         axis='columns')
  .rename(columns=rename_dict)
  .groupby(['AIR1', 'AIR2'])
  .size()
  .loc[('ATL', 'IAH')]
)

# timing it to compare to Numpy's sorting method below


2.96 s ± 91 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [90]:
(flights     # doctest: +SKIP
  [['ORG_AIR', 'DEST_AIR']]
  .apply(lambda ser:
         ser.sort_values().reset_index(drop=True),
         axis='columns')
  .rename(columns=rename_dict)
  .groupby(['AIR1', 'AIR2'])
  .size()
  .loc[('IAH', 'ATL')]
)

KeyError: ('IAH', 'ATL')

In [91]:
flights[['ORG_AIR', 'DEST_AIR']]

,ORG_AIR,DEST_AIR
0,LAX,SLC
1,DEN,IAD
2,DFW,VPS
3,DFW,DCA
4,LAX,MCI
...,...,...
58487,SFO,DFW
58488,LAS,SFO
58489,SFO,SBA
58490,MSP,ATL


In [92]:
data_sorted = np.sort(flights[['ORG_AIR', 'DEST_AIR']])
data_sorted.shape

(58492, 2)

In [93]:
data_sorted[:10]

array([['LAX', 'SLC'],
       ['DEN', 'IAD'],
       ['DFW', 'VPS'],
       ['DCA', 'DFW'],
       ['LAX', 'MCI'],
       ['IAH', 'SAN'],
       ['DFW', 'MSY'],
       ['PHX', 'SFO'],
       ['ORD', 'STL'],
       ['IAH', 'SJC']], dtype=object)

In [94]:
# In previous versions, evaluated to True . Now DataFrames are not equal
flights_sort2 = pd.DataFrame(data_sorted, columns=['AIR1', 'AIR2'])
flights_sort2.equals(f_part3.rename(columns={'ORG_AIR':'AIR1',
    'DEST_AIR':'AIR2'}))

False

In [95]:
# apply is slow and expensive => here we can use Numpy's soer function

In [96]:
%%timeit
data_sorted = np.sort(flights[['ORG_AIR', 'DEST_AIR']])
flights_sort2 = pd.DataFrame(data_sorted,
    columns=['AIR1', 'AIR2'])

5.63 ms ± 129 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Finding the longest streak of on-time flights

In [97]:
s = pd.Series([0, 1, 1, 0, 1, 1, 1, 0])
s

0    0
1    1
2    1
3    0
4    1
5    1
6    1
7    0
dtype: int64

In [98]:
s1 = s.cumsum()
s1

0    0
1    1
2    2
3    2
4    3
5    4
6    5
7    5
dtype: int64

In [99]:
s.mul(s1)

0    0
1    1
2    2
3    0
4    3
5    4
6    5
7    0
dtype: int64

In [100]:
s.mul(s1).diff()

0    NaN
1    1.0
2    1.0
3   -2.0
4    3.0
5    1.0
6    1.0
7   -5.0
dtype: float64

In [101]:
(s
    .mul(s.cumsum())
    .diff()
    .where(lambda x: x < 0)
)

0    NaN
1    NaN
2    NaN
3   -2.0
4    NaN
5    NaN
6    NaN
7   -5.0
dtype: float64

In [102]:
(s
    .mul(s.cumsum())
    .diff()
    .where(lambda x: x < 0)
    .ffill()
)

0    NaN
1    NaN
2    NaN
3   -2.0
4   -2.0
5   -2.0
6   -2.0
7   -5.0
dtype: float64

In [103]:
(s
    .mul(s.cumsum())
    .diff()
    .where(lambda x: x < 0)
    .ffill()
    .add(s.cumsum(), fill_value=0)
)

0    0.0
1    1.0
2    2.0
3    0.0
4    1.0
5    2.0
6    3.0
7    0.0
dtype: float64

In [104]:
flights = pd.read_csv('../data/flights.csv')
(flights
    .assign(ON_TIME=flights['ARR_DELAY'].lt(15).astype(int))
    [['AIRLINE', 'ORG_AIR', 'ON_TIME']]
)

,AIRLINE,ORG_AIR,ON_TIME
0,WN,LAX,0
1,UA,DEN,1
2,MQ,DFW,0
3,AA,DFW,1
4,WN,LAX,0
...,...,...,...
58487,AA,SFO,1
58488,F9,LAS,1
58489,OO,SFO,1
58490,WN,MSP,0


In [105]:
def max_streak(s):
    s1 = s.cumsum()
    return (s
       .mul(s1)
       .diff()
       .where(lambda x: x < 0)
       .ffill()
       .add(s1, fill_value=0)
       .max()
    )

In [106]:
(flights
    .assign(ON_TIME=flights['ARR_DELAY'].lt(15).astype(int))
    .sort_values(['MONTH', 'DAY', 'SCHED_DEP'])
    .groupby(['AIRLINE', 'ORG_AIR'])
    ['ON_TIME']
    .agg(['mean', 'size', max_streak])
    .round(2)
)

mean  size  max_streak
AIRLINE ORG_AIR                        
AA      ATL      0.82   233        15.0
        DEN      0.74   219        17.0
        DFW      0.78  4006        64.0
        IAH      0.80   196        24.0
        LAS      0.79   374        29.0
...               ...   ...         ...
WN      LAS      0.77  2031        39.0
        LAX      0.70  1135        23.0
        MSP      0.84   237        32.0
        PHX      0.77  1724        33.0
        SFO      0.76   445        17.0

[114 rows x 3 columns]

In [107]:
def max_delay_streak(df):
    df = df.reset_index(drop=True)
    late = 1 - df['ON_TIME']
    late_sum = late.cumsum()
    streak = (late
        .mul(late_sum)
        .diff()
        .where(lambda x: x < 0)
        .ffill()
        .add(late_sum, fill_value=0)
    )
    last_idx = streak.idxmax()
    first_idx = last_idx - streak.max() + 1
    res = (df
        .loc[[first_idx, last_idx], ['MONTH', 'DAY']]
        .assign(streak=streak.max())
    )
    res.index = ['first', 'last']
    return res

In [108]:
(flights
    .assign(ON_TIME=flights['ARR_DELAY'].lt(15).astype(int))
    .sort_values(['MONTH', 'DAY', 'SCHED_DEP'])
    .groupby(['AIRLINE', 'ORG_AIR'])
    .apply(max_delay_streak)
    .sort_values('streak', ascending=False)
)

KeyError: '[1.0] not in index'